In [1]:
# Cell 1: Setup and load policy (run once)
from forge.rl import Policy
from vllm.transformers_utils.tokenizer import get_tokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = get_tokenizer(model_name)

# Load policy once - await works directly in Jupyter
policy = await Policy.options(
    procs=1,
    num_replicas=1,
    with_gpus=True,
).as_service(
    engine_args={"model": model_name},
    sampling_params={"n": 1, "max_tokens": 2048},
)
print("Policy loaded!")

/root/anaconda3/envs/forge/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 12-12 19:41:45 [__init__.py:235] Automatically detected platform cuda.
Spawning service Generator
Launcher not provided, remote allocations will not work.
INFO 12-12 19:41:58 [__init__.py:235] Automatically detected platform cuda.


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 12-12 19:42:06 [config.py:1604] Using max model len 40960
INFO 12-12 19:42:06 [config.py:2434] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 12-12 19:42:09 [__init__.py:235] Automatically detected platform cuda.
WARNING 12-12 19:42:10 [multiproc_worker_utils.py:307] Reducing Torch parallelism from 32 threads to 1 to avoid unnecessary CPU contention. Set OMP_NUM_THREADS in the external environment to tune this value as needed.


[W1212 19:42:13.704807262 ProcessGroupNCCL.cpp:924] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


...

INFO 12-12 19:42:14 [default_loader.py:262] Loading weights took 0.64 seconds
INFO 12-12 19:42:15 [gpu_model_runner.py:1892] Model loading took 3.2152 GiB and 1.352315 seconds
INFO 12-12 19:42:21 [backends.py:530] Using cache directory: /root/.cache/vllm/torch_compile_cache/5417a2c099/rank_0_0/backbone for vLLM's torch.compile
INFO 12-12 19:42:21 [backends.py:541] Dynamo bytecode transform time: 4.56 s
INFO 12-12 19:42:23 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.031 s
INFO 12-12 19:42:26 [monitor.py:34] torch.compile takes 4.56 s in total
INFO 12-12 19:42:26 [gpu_worker.py:255] Available KV cache memory: 66.67 GiB
INFO 12-12 19:42:27 [kv_cache_utils.py:833] GPU KV cache size: 624,224 tokens
INFO 12-12 19:42:27 [kv_cache_utils.py:837] Maximum concurrency for 40,960 tokens per request: 15.24x


Capturing CUDA graph shapes: 100%|██████████| 67/67 [00:01<00:00, 39.88it/s]


INFO 12-12 19:42:29 [gpu_model_runner.py:2485] Graph capturing finished in 2 secs, took 0.45 GiB


[-]E1212 19:42:36.426220 481383 hyperactor/src/channel/net.rs:872] error_msg:session unix:@LP9pLtUqNpm0dpxcBxQCISO6.15933586513064632734: failed to deliver message within timeout


Policy loaded!


In [2]:
# Cell 2: Use the policy interactively (run as many times as you want)


tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical equation. Uses Python's eval() to compute the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "equation": {
                        "type": "string",
                        "description": "The mathematical equation to evaluate, e.g. '2 + 2' or '(3 * 4) / 2'",
                    },
                },
                "required": ["equation"],
                "additionalProperties": False,
            },
        },
    },
]

messages = [
    {"role": "system", "content": "You are a helpful assistant that can evaluate mathematical equations."},
    {"role": "user", "content": "What is 2 + 2?"},
]

formatted_request = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

response = await policy.generate.route(formatted_request)
print("="*100)
print(response[0].text)
print("="*100)

`mlogger = await get_or_create_metric_logger(process_name='Controller')`
`await mlogger.init_backends.call_one(logging_config)`
or set env variable `FORGE_DISABLE_METRICS=True`


[Generator-0/1] 2025-12-12 19:42:46 WARNING Skipping metric collection for Generator_19QSmWC4mEym_r0. Metric logging backends (e.g. wandb) were not initialized. This happens when you try to use `record_metric` before calling `init_backends`. To disable this warning, please call in your main file:
`mlogger = await get_or_create_metric_logger(process_name='Controller')`
`await mlogger.init_backends.call_one(logging_config)`
or set env variable `FORGE_DISABLE_METRICS=True`
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform cuda.
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform cuda.
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform cuda.
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform cuda.
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform cuda.
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform cuda.
INFO 12-12 19:42:46 [__init__.py:235] Automatically detected platform